# Plant Disease Classification (38 classes): MobileNetV2 / EfficientNetB0

Faster and more accurate training pipeline:
- GPU-side augmentation and preprocessing inside the model (no CPU `ImageDataGenerator` bottleneck)
- `tf.data` with prefetch, validation set cached in RAM
- Mixed precision (T4 tensor cores)
- Two-phase transfer learning: train the head, then fine-tune the top backbone layers at a low learning rate
- Label smoothing, learning-rate reduction on plateau, early stopping
- Evaluation on the full validation set, with measured parameters, size and latency

**Runtime > Change runtime type > T4 GPU** before running. To compare models, run once with `MODEL_NAME='mobilenetv2'`, then again with `'efficientnetb0'`.

## 1. Setup and configuration

In [ ]:
import os, json, time, zipfile, random, glob, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

# ---------------- Config ----------------
MODEL_NAME      = 'efficientnetb0' 
IMG_SIZE        = (224, 224)
BATCH_SIZE      = 64
WARMUP_EPOCHS   = 3               # phase 1: backbone frozen
FINETUNE_EPOCHS = 8               # phase 2: top backbone layers unfrozen
FINETUNE_LAYERS = 60              # number of top backbone layers to unfreeze
WEIGHTS         = 'imagenet'
OUT             = '/content'
SEED            = 42
# -----------------------------------------

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else 'NONE (enable a GPU runtime for fast training)')
if gpus:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

for d in ['models', 'results', 'figures']:
    os.makedirs(f'{OUT}/{d}', exist_ok=True)
print('TensorFlow', tf.__version__)

## 2. Download the dataset

In [ ]:
try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub

try:
    DATA_ROOT = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')
except Exception as exc:
    print('Download failed (' + str(exc)[:120] + '). Logging in to Kaggle...')
    kagglehub.login()
    DATA_ROOT = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')
print('Dataset path:', DATA_ROOT)

## 3. Locate train and validation folders

In [ ]:
def find_split_dirs(root):
    train_dir = valid_dir = None
    for dirpath, dirnames, _ in os.walk(root):
        base = os.path.basename(dirpath).lower()
        if len(dirnames) == 38:                      # 38 class folders
            if base == 'train':
                train_dir = dirpath
            elif base in ('valid', 'validation'):
                valid_dir = dirpath
    return train_dir, valid_dir

TRAIN_DIR, VALID_DIR = find_split_dirs(DATA_ROOT)
assert TRAIN_DIR and VALID_DIR, 'Could not find train/valid folders with 38 classes'
print('Train:', TRAIN_DIR)
print('Valid:', VALID_DIR)

## 4. Input pipeline

Validation images are stored as uint8 in RAM after the first epoch (about 2.6 GB), so later epochs read them instantly.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=True, seed=SEED)

valid_raw = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=False)

class_names = train_raw.class_names
assert class_names == valid_raw.class_names
NUM_CLASSES = len(class_names)
valid_paths = valid_raw.file_paths
print('Classes:', NUM_CLASSES, '| train images:', len(train_raw.file_paths), '| valid images:', len(valid_paths))

opts = tf.data.Options()
opts.deterministic = False
train_ds = train_raw.with_options(opts).prefetch(AUTOTUNE)

valid_ds = (valid_raw
            .map(lambda x, y: (tf.cast(x, tf.uint8), y), num_parallel_calls=AUTOTUNE)
            .cache()
            .map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))

## 5. Model

Augmentation (flip, rotation, zoom) and scaling run on the GPU inside the model and are inactive at inference. The model takes raw 0-255 pixels.

In [ ]:
def build_model(name, num_classes):
    if name == 'mobilenetv2':
        backbone = tf.keras.applications.MobileNetV2(
            input_shape=IMG_SIZE + (3,), include_top=False, weights=WEIGHTS)
    elif name == 'efficientnetb0':
        backbone = tf.keras.applications.EfficientNetB0(
            input_shape=IMG_SIZE + (3,), include_top=False, weights=WEIGHTS)
    else:
        raise ValueError('MODEL_NAME must be mobilenetv2 or efficientnetb0')
    backbone.trainable = False

    augment = tf.keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ], name='augment')

    inputs = Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    if name == 'mobilenetv2':
        x = layers.Rescaling(1 / 127.5, offset=-1)(x)   # [-1, 1]; EfficientNet scales internally
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    return Model(inputs, outputs, name=name), backbone

def compile_model(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy'])

model, backbone = build_model(MODEL_NAME, NUM_CLASSES)
model_path = f'{OUT}/models/{MODEL_NAME}.keras'
print('Total parameters: %.2fM' % (model.count_params() / 1e6))

## 6. Phase 1: train the classifier head (backbone frozen)

In [ ]:
t_start = time.time()
compile_model(model, 1e-3)
h1 = model.fit(train_ds, validation_data=valid_ds, epochs=WARMUP_EPOCHS, verbose=1)

## 7. Phase 2: fine-tune the top backbone layers

BatchNorm layers stay frozen. The best checkpoint by validation accuracy is saved.

In [ ]:
backbone.trainable = True
for layer in backbone.layers[:-FINETUNE_LAYERS]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

compile_model(model, 1e-4)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True),
    ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=1, min_lr=1e-6),
]

h2 = model.fit(train_ds, validation_data=valid_ds,
               epochs=WARMUP_EPOCHS + FINETUNE_EPOCHS, initial_epoch=WARMUP_EPOCHS,
               callbacks=callbacks, verbose=1)

train_minutes = (time.time() - t_start) / 60
print('Total training time: %.1f min' % train_minutes)

keys = ['loss', 'accuracy', 'val_loss', 'val_accuracy']
history = {k: h1.history[k] + h2.history[k] for k in keys}
pd.DataFrame(history).to_csv(f'{OUT}/results/{MODEL_NAME}_history.csv', index_label='epoch')

## 8. Training curves

In [ ]:
epochs = np.arange(1, len(history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, key, title in [(axes[0], 'loss', 'Loss'), (axes[1], 'accuracy', 'Accuracy')]:
    ax.plot(epochs, history[key], marker='o', linewidth=2, label='Train')
    ax.plot(epochs, history['val_' + key], marker='s', linewidth=2, label='Validation')
    ax.axvline(WARMUP_EPOCHS + 0.5, color='gray', linestyle='--', label='Fine-tuning starts')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig(f'{OUT}/figures/{MODEL_NAME}_training_curves.png', dpi=180, bbox_inches='tight')
plt.show()

## 9. Evaluation on the full validation set

In [ ]:
best = tf.keras.models.load_model(model_path)
probs = best.predict(valid_ds, verbose=1)
y_pred = probs.argmax(axis=1)
conf = probs.max(axis=1)
y_true = np.array([class_names.index(os.path.basename(os.path.dirname(p))) for p in valid_paths])

metrics = {
    'Accuracy':           accuracy_score(y_true, y_pred),
    'Macro Precision':    precision_score(y_true, y_pred, average='macro', zero_division=0),
    'Macro Recall':       recall_score(y_true, y_pred, average='macro', zero_division=0),
    'Macro F1':           f1_score(y_true, y_pred, average='macro', zero_division=0),
    'Weighted Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
    'Weighted Recall':    recall_score(y_true, y_pred, average='weighted', zero_division=0),
    'Weighted F1':        f1_score(y_true, y_pred, average='weighted', zero_division=0),
}
metrics_df = pd.DataFrame(metrics.items(), columns=['Metric', 'Score'])
metrics_df.to_csv(f'{OUT}/results/{MODEL_NAME}_overall_metrics.csv', index=False)
display(metrics_df.style.format({'Score': '{:.2%}'}))

report = classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                               target_names=class_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(f'{OUT}/results/{MODEL_NAME}_classification_report.csv')

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(20, 18))
sns.heatmap(cm, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix ({MODEL_NAME})')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(f'{OUT}/figures/{MODEL_NAME}_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()

## 10. Per-class F1 and difficult classes

In [ ]:
per_class = report_df.loc[class_names].sort_values('f1-score')

plt.figure(figsize=(12, 10))
plt.barh(per_class.index, per_class['f1-score'], color='#2E86AB')
plt.xlabel('F1-score')
plt.title('Per-Class F1 Scores')
plt.tight_layout()
plt.savefig(f'{OUT}/figures/{MODEL_NAME}_per_class_f1.png', dpi=180, bbox_inches='tight')
plt.show()

print('Lowest 5 classes:')
display(per_class.head(5)[['precision', 'recall', 'f1-score', 'support']])
print('Highest 5 classes:')
display(per_class.tail(5)[['precision', 'recall', 'f1-score', 'support']])

## 11. Error analysis

In [ ]:
wrong = np.where(y_true != y_pred)[0]
error_df = pd.DataFrame({
    'path': [valid_paths[i] for i in wrong],
    'true_class': [class_names[y_true[i]] for i in wrong],
    'predicted_class': [class_names[y_pred[i]] for i in wrong],
    'confidence': conf[wrong],
}).sort_values('confidence', ascending=False)
error_df.to_csv(f'{OUT}/results/{MODEL_NAME}_errors.csv', index=False)

print('Incorrect predictions:', len(wrong), 'of', len(y_true), '| error rate: %.2f%%' % (100 * len(wrong) / len(y_true)))
print('High-confidence (>80%) errors:', int((conf[wrong] > 0.80).sum()))

print('Most confused class pairs:')
pairs = error_df.groupby(['true_class', 'predicted_class']).size().sort_values(ascending=False).head(8)
display(pairs.rename('count').reset_index())
display(error_df.head(10))

## 12. Efficiency: parameters, model size, inference latency

Latency is measured here (batch size 1, same device as training), not copied from elsewhere.

In [ ]:
sample = (np.random.rand(1, *IMG_SIZE, 3) * 255).astype('float32')
for _ in range(10):
    best(sample, training=False).numpy()                     # warm-up
n = 100
t0 = time.perf_counter()
for _ in range(n):
    best(sample, training=False).numpy()
latency_ms = (time.perf_counter() - t0) / n * 1000

device = tf.config.list_physical_devices('GPU')
efficiency = {
    'model': MODEL_NAME,
    'accuracy': metrics['Accuracy'],
    'macro_f1': metrics['Macro F1'],
    'weighted_f1': metrics['Weighted F1'],
    'params_m': best.count_params() / 1e6,
    'size_mb': os.path.getsize(model_path) / 1e6,
    'latency_ms': latency_ms,
    'train_minutes': train_minutes,
    'device': 'GPU' if device else 'CPU',
}
with open(f'{OUT}/results/{MODEL_NAME}_metrics.json', 'w') as f:
    json.dump(efficiency, f, indent=2)
display(pd.DataFrame([efficiency]))

## 13. Model comparison

Shows every model trained so far in this session or folder. Run the notebook again with `MODEL_NAME = 'efficientnetb0'` to add it.

In [ ]:
rows = [json.load(open(p)) for p in sorted(glob.glob(f'{OUT}/results/*_metrics.json'))]
comparison = pd.DataFrame(rows)
comparison.to_csv(f'{OUT}/results/model_comparison.csv', index=False)
display(comparison.style.format({'accuracy': '{:.2%}', 'macro_f1': '{:.2%}', 'weighted_f1': '{:.2%}',
                                 'params_m': '{:.2f}', 'size_mb': '{:.1f}', 'latency_ms': '{:.1f}',
                                 'train_minutes': '{:.1f}'}))

if len(comparison) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    x = np.arange(len(comparison))
    axes[0].bar(x - 0.2, comparison['accuracy'], width=0.4, label='Accuracy')
    axes[0].bar(x + 0.2, comparison['macro_f1'], width=0.4, label='Macro F1')
    axes[0].set_xticks(x); axes[0].set_xticklabels(comparison['model'])
    axes[0].set_ylim(0, 1); axes[0].set_title('Classification performance'); axes[0].legend()
    axes[1].bar(comparison['model'], comparison['params_m'], color='#A23B72')
    axes[1].set_title('Parameters (millions)')
    axes[2].bar(comparison['model'], comparison['latency_ms'], color='#2E86AB')
    axes[2].set_title('Inference latency (ms, batch 1)')
    plt.tight_layout()
    plt.savefig(f'{OUT}/figures/model_comparison.png', dpi=180, bbox_inches='tight')
    plt.show()

## 14. Predict on a single image

Upload a JPEG or PNG leaf image to see the top-5 predictions.

In [ ]:
from google.colab import files as colab_files

uploaded = colab_files.upload()
for name, content in uploaded.items():
    img_path = f'{OUT}/{name}'
    with open(img_path, 'wb') as f:
        f.write(content)
    image = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(image)
    p = best.predict(np.expand_dims(arr, 0), verbose=0)[0]
    top = p.argsort()[-5:][::-1]

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'{class_names[top[0]]} ({p[top[0]]:.1%})')
    plt.show()
    display(pd.DataFrame({'Class': [class_names[i] for i in top], 'Probability': [p[i] for i in top]}))

## 15. Export results

In [ ]:
zip_path = f'{OUT}/Plant_Disease_Results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in ['results', 'figures', 'models']:
        for root, dirs, fnames in os.walk(f'{OUT}/{folder}'):
            for fname in fnames:
                full = os.path.join(root, fname)
                archive.write(full, os.path.relpath(full, OUT))
print('Export:', zip_path)

from google.colab import files as colab_files
colab_files.download(zip_path)